# Carga de datos

In [1]:
import os, random
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, log_loss, accuracy_score, average_precision_score, f1_score
from sentence_transformers import SentenceTransformer

from sklearn.model_selection import train_test_split

from deepctr_torch.inputs import SparseFeat, DenseFeat, get_feature_names
from deepctr_torch.models import DeepFM
from deepctr_torch.callbacks import EarlyStopping

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = 'cpu'

c:\Users\Casa\Desktop\U\10mo Semestre\Sistemas Recomendadores\Proyecto\Proyecto_RecSys\modelo_deepfm\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
games = pd.read_csv("Game Recommendations on Steam/games.csv")
recs = pd.read_csv("Game Recommendations on Steam/recommendations.csv")
desc = pd.read_csv("Steam Store Games/steam_description_data.csv")
media = pd.read_csv("Steam Store Games/steam_media_data.csv")
matched = pd.read_csv("games_steam_matched.csv")

In [3]:
desc = desc.rename(columns={"steam_appid": "appid"})
media = media.rename(columns={"steam_appid": "appid"})
matched = matched.merge(desc[['appid', 'about_the_game']], on='appid', how='left')
matched = matched.merge(media[['appid', 'header_image']], on='appid', how='left')
df_merged = matched 

In [4]:
df_merged["game_title"] = df_merged["title"].fillna(df_merged["name"])
df_merged = df_merged.drop(columns=["title", "name"], errors="ignore")

In [5]:
train_split = pd.read_csv("data3/split/train_split.csv")
val_split   = pd.read_csv("data3/split/val_split.csv")
test_split  = pd.read_csv("data3/split/test_split.csv")
sampled_recs = pd.read_csv("data3/sampled_recommendations.csv")

In [6]:
train_merged = train_split.merge(df_merged, on="app_id", how="left")
val_merged   = val_split.merge(df_merged, on="app_id", how="left")
test_merged  = test_split.merge(df_merged, on="app_id", how="left")
sampled_recs_merged  = sampled_recs.merge(df_merged, on="app_id", how="left")

In [7]:
target = ['is_recommended']

sparse_features = [
    'user_id',   
    'rating',
]

dense_features = [
    'price_final',
    'average_playtime',
]

In [8]:
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import numpy as np

sampled_recs_merged[sparse_features] = sampled_recs_merged[sparse_features].fillna('-1')
sampled_recs_merged[dense_features] = sampled_recs_merged[dense_features].fillna(0)

for feat in sparse_features:
    lbe = LabelEncoder()
    sampled_recs_merged[feat] = lbe.fit_transform(sampled_recs_merged[feat].astype(str))

mms = MinMaxScaler(feature_range=(0, 1))
sampled_recs_merged[dense_features] = mms.fit_transform(sampled_recs_merged[dense_features])

In [9]:
sampled_recs_merged["text_input"] = (
    sampled_recs_merged["about_the_game"]
    .fillna(sampled_recs_merged["game_title"])
    .fillna("")
)

# Embedding de texto = AllMiniLM

In [10]:
from sentence_transformers import SentenceTransformer
import numpy as np

ejecutar_embedding = False
if ejecutar_embedding:
    text_model = SentenceTransformer("all-MiniLM-L6-v2")  

    texts = sampled_recs_merged["text_input"].tolist()

    text_emb = text_model.encode(
        texts,
        batch_size=64,
        convert_to_numpy=True,
        show_progress_bar=True
    )
    emb_dim = text_emb.shape[1]
    print("Dimensión embedding de texto:", emb_dim)


Cargar Embedding

In [11]:
text_emb_all_mini = np.load("text_emb_sampled3.npy")
emb_dim_all_mini = text_emb_all_mini.shape[1]

In [12]:
appid2idx = {app_id: i for i, app_id in enumerate(sampled_recs_merged["app_id"])}

# Embedding de texto = mpnet

In [13]:
ejecutar_embedding = False
if ejecutar_embedding:
    text_model = SentenceTransformer('all-mpnet-base-v2') 

    texts = sampled_recs_merged["text_input"].tolist()

    text_emb = text_model.encode(
        texts,
        batch_size=64,
        convert_to_numpy=True,
        show_progress_bar=True
    )
    emb_dim = text_emb.shape[1]
    print("Dimensión embedding de texto:", emb_dim)

Cargar Embedding

In [14]:
text_emb_mpnet = np.load("text_emb_all-mpnet-base-v2.npy")
emb_dim_mpnet = text_emb_mpnet.shape[1] 

In [15]:
id2idx = {rid: i for i, rid in enumerate(sampled_recs_merged["review_id"])}

In [16]:
appid2idx = {app_id: i for i, app_id in enumerate(sampled_recs_merged["app_id"])}

# Embedding imagen con CLIP

In [17]:
import clip
import torch
from PIL import Image
import requests
from io import BytesIO
import numpy as np
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model_clip, preprocess = clip.load("ViT-B/32", device=device)

def get_image_embedding(url):
    try:
        if not isinstance(url, str) or url.strip() == "":
            raise ValueError("empty url")
        resp = requests.get(url, timeout=5)
        resp.raise_for_status()
        image = Image.open(BytesIO(resp.content)).convert("RGB")
        img = preprocess(image).unsqueeze(0).to(device)

        with torch.no_grad():
            emb = model_clip.encode_image(img)

        emb = emb / emb.norm(dim=-1, keepdim=True) 
        return emb.cpu().numpy().flatten()
    except Exception:
        return np.zeros((512,), dtype="float32")  



Device: cpu


In [18]:
guardar_embedding_img = False   
if guardar_embedding_img: 
    img_emb_list = []

    for url in tqdm(sampled_recs_merged["header_image"], desc="Imagenes"):
        emb = get_image_embedding(url)
        img_emb_list.append(emb)

    img_emb = np.vstack(img_emb_list).astype("float32")  
    img_emb_dim = img_emb.shape[1]
    print("Shape img_emb:", img_emb.shape)

    np.save("clip_img_emb_by_app.npy", img_emb)


In [19]:
img_emb = np.load("clip_img_emb_by_app.npy")
img_emb_dim = img_emb.shape[1]

In [20]:
appid2idx = {app_id: i for i, app_id in enumerate(sampled_recs_merged["app_id"])}

In [21]:
def get_split_img_emb(split_df, appid2idx, img_emb):
    idxs = [appid2idx[aid] for aid in split_df["app_id"]]
    return img_emb[idxs]

train_img_emb = get_split_img_emb(train_merged, appid2idx, img_emb)
val_img_emb   = get_split_img_emb(val_merged, appid2idx, img_emb)
test_img_emb  = get_split_img_emb(test_merged, appid2idx, img_emb)

print(train_img_emb.shape, val_img_emb.shape, test_img_emb.shape)

(63905, 512) (11649, 512) (12784, 512)


# Funciones Transversales

In [22]:
def build_candidates_rank_df(model, test_df, user_seen, all_items, 
                             catalog_df,
                             id2idx, text_emb, feature_names,
                             appid2idx, img_emb, dense_features,
                             K=10, max_candidates=500):
    rows = []

    for u, df_u in test_df.groupby("user_id"):
        seen_u = user_seen.get(u, set())
        pos_items = set(df_u[df_u["is_recommended"] == 1]["app_id"].tolist())

        candidates = np.setdiff1d(all_items, np.fromiter(seen_u, dtype=int), assume_unique=True)

        if len(candidates) == 0 or len(pos_items) == 0:
            continue

        if max_candidates is not None and len(candidates) > max_candidates:
            rng = np.random.default_rng(42)
            sampled_neg = set(rng.choice(candidates, size=max_candidates, replace=False))
        else:
            sampled_neg = set(candidates)

        items_eval = list(pos_items.union(sampled_neg))

        df_eval = pd.DataFrame({
            "user_id": [u] * len(items_eval),
            "app_id": items_eval,
        })

        df_eval = df_eval.merge(
            test_df[["user_id", "app_id", "is_recommended", "rating", "review_id"]],
            on=["user_id", "app_id"],
            how="left"
        )

        df_eval = df_eval.merge(
            catalog_df,
            on="app_id",
            how="left"
        )

        df_eval["is_recommended"] = df_eval["is_recommended"].fillna(0).astype(int)
        df_eval["rating"] = df_eval["rating"].fillna(0).astype("int32")
        df_eval["price_final"] = df_eval["price_final"].fillna(0).astype("float32")
        df_eval["average_playtime"] = df_eval["average_playtime"].fillna(0).astype("float32")
        df_eval["user_id"] = df_eval["user_id"].astype("int32")

        # --- Texto: solo si text_emb NO es None ---
        if (text_emb is not None) and ("review_id" in df_eval.columns):
            idxs = []
            for rid in df_eval["review_id"]:
                if pd.isna(rid) or rid not in id2idx:
                    idxs.append(None)
                else:
                    idxs.append(id2idx[rid])

            text_emb_local = np.zeros((len(df_eval), text_emb.shape[1]), dtype=text_emb.dtype)
            valid_idx = [i for i, idx in enumerate(idxs) if idx is not None]
            if valid_idx:
                text_emb_local[valid_idx] = text_emb[
                    [id2idx[df_eval["review_id"].iloc[i]] for i in valid_idx]
                ]
        else:
            text_emb_local = None  # no se usa texto

        # --- Imágenes (igual que antes) ---
        if img_emb is not None:
            img_idxs = []
            for aid in df_eval["app_id"]:
                if aid in appid2idx:
                    img_idxs.append(appid2idx[aid])
                else:
                    img_idxs.append(None)

            img_emb_local = np.zeros((len(df_eval), img_emb.shape[1]), dtype=img_emb.dtype)
            valid_img_idx = [i for i, idx in enumerate(img_idxs) if idx is not None]
            if valid_img_idx:
                img_emb_local[valid_img_idx] = img_emb[
                    [appid2idx[df_eval["app_id"].iloc[i]] for i in valid_img_idx]
                ]
        else:
            img_emb_local = None

        for feat in dense_features:
            if feat in df_eval.columns:
                df_eval[feat] = df_eval[feat].astype("float32")

        # Construcción de X_rank
        X_rank = {
            name: df_eval[name].values
            for name in feature_names
            if name not in ["text_emb", "img_emb"]
        }

        if text_emb_local is not None:
            X_rank["text_emb"] = text_emb_local
        if img_emb_local is not None:
            X_rank["img_emb"] = img_emb_local

        scores = model.predict(X_rank, batch_size=256).reshape(-1)

        tmp = pd.DataFrame({
            "user_id": df_eval["user_id"].values,
            "app_id": df_eval["app_id"].values,
            "label": df_eval["is_recommended"].astype(int).values,
            "score": scores,
        })
        if "genres" in df_eval.columns:
            tmp["genres"] = df_eval["genres"].values

        rows.append(tmp)

    if not rows:
        return pd.DataFrame(columns=["user_id", "app_id", "label", "score"])

    rank_df = pd.concat(rows, ignore_index=True)
    rank_df = rank_df.sort_values(["user_id", "score"], ascending=[True, False]).reset_index(drop=True)
    return rank_df

In [23]:
def build_user_seen(train_df, val_df):
    seen = {}
    for df in [train_df, val_df]:
        for u, group in df.groupby("user_id"):
            items_u = set(group["app_id"].tolist())
            if u in seen:
                seen[u].update(items_u)
            else:
                seen[u] = items_u
    return seen

In [24]:
def get_split_text_emb(split_df, id2idx, text_emb):
    idxs = [id2idx[rid] for rid in split_df["review_id"]]
    return text_emb[idxs]

In [25]:
def correr_modelo_texto(text_emb, emb_dim):
    fixlen_feature_columns = (
        [
            SparseFeat(
                feat,
                vocabulary_size=sampled_recs_merged[feat].nunique(),
                embedding_dim=8
            )
            for feat in sparse_features
        ]
        +
        [DenseFeat(feat, 1) for feat in dense_features]
        +
        [DenseFeat("text_emb", emb_dim)]      
    )

    dnn_feature_columns = fixlen_feature_columns
    linear_feature_columns = fixlen_feature_columns

    feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)

    id2idx = {rid: i for i, rid in enumerate(sampled_recs_merged["review_id"])}

    train_text_emb = get_split_text_emb(train_merged, id2idx, text_emb)
    val_text_emb   = get_split_text_emb(val_merged, id2idx, text_emb)
    test_text_emb  = get_split_text_emb(test_merged, id2idx, text_emb)

    train_merged[sparse_features] = train_merged[sparse_features].fillna('-1')
    val_merged[sparse_features]   = val_merged[sparse_features].fillna('-1')
    test_merged[sparse_features]  = test_merged[sparse_features].fillna('-1')

    for feat in sparse_features:
        lbe = LabelEncoder()
        all_vals = pd.concat([
            train_merged[feat],
            val_merged[feat],
            test_merged[feat]
        ], axis=0).astype(str)
        lbe.fit(all_vals)
        train_merged[feat] = lbe.transform(train_merged[feat].astype(str))
        val_merged[feat]   = lbe.transform(val_merged[feat].astype(str))
        test_merged[feat]  = lbe.transform(test_merged[feat].astype(str))

    for feat in dense_features:
        train_merged[feat] = pd.to_numeric(train_merged[feat], errors='coerce')
        val_merged[feat]   = pd.to_numeric(val_merged[feat], errors='coerce')
        test_merged[feat]  = pd.to_numeric(test_merged[feat], errors='coerce')

        train_merged[feat] = train_merged[feat].fillna(0)
        val_merged[feat]   = val_merged[feat].fillna(0)
        test_merged[feat]  = test_merged[feat].fillna(0)
    for df in [train_merged, val_merged, test_merged]:
        df["is_recommended"] = df["is_recommended"].astype(int)

    for feat in sparse_features:
        train_merged[feat] = train_merged[feat].astype('int32')
        val_merged[feat]   = val_merged[feat].astype('int32')
        test_merged[feat]  = test_merged[feat].astype('int32')
    for feat in dense_features:
        train_merged[feat] = train_merged[feat].astype('float32')
        val_merged[feat]   = val_merged[feat].astype('float32')
        test_merged[feat]  = test_merged[feat].astype('float32')
    train_text_emb = train_text_emb.astype('float32')
    val_text_emb   = val_text_emb.astype('float32')
    test_text_emb  = test_text_emb.astype('float32')
    y_train = train_merged[target].values.astype('float32')
    y_val   = val_merged[target].values.astype('float32')
    y_test  = test_merged[target].values.astype('float32')

    train_model_input = {
        name: train_merged[name].values
        for name in feature_names
        if name not in ["text_emb"]
    }
    train_model_input["text_emb"] = train_text_emb.astype("float32")

    val_model_input = {
        name: val_merged[name].values
        for name in feature_names
        if name not in ["text_emb"]
    }
    val_model_input["text_emb"] = val_text_emb.astype("float32")

    test_model_input = {
        name: test_merged[name].values
        for name in feature_names
        if name not in ["text_emb"]
    }
    test_model_input["text_emb"] = test_text_emb.astype("float32")

    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = DeepFM(
        linear_feature_columns=linear_feature_columns,
        dnn_feature_columns=dnn_feature_columns,
        task="binary",
        device=device,
    )

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["auc"],
    )

    model.fit(
        train_model_input,
        y_train,
        batch_size=256,
        epochs=10,
        verbose=2,
        validation_data=(val_model_input, y_val),
    )

    pred_test = model.predict(test_model_input, batch_size=256)
    return pred_test, y_test, model, feature_names

# Modelo solo con texto =  All Mini

In [50]:
text_emb_all_mini = text_emb_all_mini
emb_dim = text_emb_all_mini.shape[1]
y_pred_allmini, y_test_allmini, model_allmini, feature_names_allmini = correr_modelo_texto(text_emb_all_mini, emb_dim)
from sklearn.metrics import roc_auc_score, log_loss
print("AUC test:", roc_auc_score(y_test_allmini, y_pred_allmini))
print("LogLoss test:", log_loss(y_test_allmini, y_pred_allmini))

cpu
Train on 63905 samples, validate on 11649 samples, 250 steps per epoch
Epoch 1/10
2s - loss:  0.4890 - auc:  0.5527 - val_auc:  0.6718
Epoch 2/10
2s - loss:  0.3675 - auc:  0.7411 - val_auc:  0.6737
Epoch 3/10
2s - loss:  0.4216 - auc:  0.7952 - val_auc:  0.7040
Epoch 4/10
2s - loss:  0.3725 - auc:  0.8312 - val_auc:  0.7178
Epoch 5/10
2s - loss:  0.2805 - auc:  0.8710 - val_auc:  0.7097
Epoch 6/10
2s - loss:  0.2837 - auc:  0.8758 - val_auc:  0.7029
Epoch 7/10
2s - loss:  0.2561 - auc:  0.8938 - val_auc:  0.7060
Epoch 8/10
2s - loss:  0.2619 - auc:  0.8992 - val_auc:  0.7056
Epoch 9/10
2s - loss:  0.2537 - auc:  0.9024 - val_auc:  0.7020
Epoch 10/10
2s - loss:  0.2414 - auc:  0.9120 - val_auc:  0.7082
AUC test: 0.7005456835923574
LogLoss test: 0.5658085217440303


In [51]:
user_seen = build_user_seen(train_merged, val_merged)
item_cols = ["app_id", "price_final", "average_playtime"]
if "genres" in df_merged.columns:
    item_cols.append("genres")
catalog_df = df_merged[item_cols].drop_duplicates(subset=["app_id"]).copy()

In [52]:
ejecutar_rank = True
id2idx = {rid: i for i, rid in enumerate(sampled_recs_merged["review_id"])}
appid2idx = {app_id: i for i, app_id in enumerate(sampled_recs_merged["app_id"])}
if ejecutar_rank == True:
    all_items = df_merged["app_id"].unique()
    rank_df_only_allmini = build_candidates_rank_df(
        model=model_allmini,
        test_df=test_merged,
        user_seen=user_seen,
        all_items=all_items,
        catalog_df=catalog_df,       # <--- Pasamos el catálogo
        id2idx=id2idx,
        text_emb=text_emb_all_mini,
        feature_names=feature_names_allmini,
        appid2idx=appid2idx,
        img_emb=None,  # <--- No usamos embeddings de imagen aquí
        dense_features=dense_features, # <--- Pasamos las features densas
        K=10,
        max_candidates=None          # <--- None para evaluar contra TODOS (sin atajos)
    )

# Modelo solo con texto = mpnet

In [53]:
text_emb_mpnet = text_emb_mpnet
emb_dim = text_emb_mpnet.shape[1]

y_pred_mpnet, y_test_mpnet, model_mpnet, feature_names_mpnet = correr_modelo_texto(text_emb_mpnet, emb_dim)
from sklearn.metrics import roc_auc_score, log_loss
print("AUC test:", roc_auc_score(y_test_mpnet, y_pred_mpnet))
print("LogLoss test:", log_loss(y_test_mpnet, y_pred_mpnet))

cpu
Train on 63905 samples, validate on 11649 samples, 250 steps per epoch
Epoch 1/10
3s - loss:  0.4428 - auc:  0.5617 - val_auc:  0.6583
Epoch 2/10
3s - loss:  0.3760 - auc:  0.7367 - val_auc:  0.6809
Epoch 3/10
3s - loss:  0.3465 - auc:  0.8105 - val_auc:  0.7123
Epoch 4/10
4s - loss:  0.3004 - auc:  0.8498 - val_auc:  0.7034
Epoch 5/10
3s - loss:  0.2955 - auc:  0.8628 - val_auc:  0.7175
Epoch 6/10
3s - loss:  0.2648 - auc:  0.8824 - val_auc:  0.7165
Epoch 7/10
3s - loss:  0.2662 - auc:  0.8833 - val_auc:  0.6984
Epoch 8/10
3s - loss:  0.2699 - auc:  0.8842 - val_auc:  0.7183
Epoch 9/10
3s - loss:  0.2606 - auc:  0.8913 - val_auc:  0.7087
Epoch 10/10
3s - loss:  0.2491 - auc:  0.8977 - val_auc:  0.7126
AUC test: 0.7080628385060626
LogLoss test: 0.61082253174717


In [54]:
user_seen = build_user_seen(train_merged, val_merged)
item_cols = ["app_id", "price_final", "average_playtime"]
if "genres" in df_merged.columns:
    item_cols.append("genres")
catalog_df = df_merged[item_cols].drop_duplicates(subset=["app_id"]).copy()

In [55]:
ejecutar_rank = True
id2idx = {rid: i for i, rid in enumerate(sampled_recs_merged["review_id"])}
appid2idx = {app_id: i for i, app_id in enumerate(sampled_recs_merged["app_id"])}
if ejecutar_rank == True:
    all_items = df_merged["app_id"].unique()
    rank_df_only_mpnet = build_candidates_rank_df(
        model=model_mpnet,
        test_df=test_merged,
        user_seen=user_seen,
        all_items=all_items,
        catalog_df=catalog_df,       # <--- Pasamos el catálogo
        id2idx=id2idx,
        text_emb=text_emb_mpnet,
        feature_names=feature_names_mpnet,
        appid2idx=appid2idx,
        img_emb=None,  # <--- No usamos embeddings de imagen aquí
        dense_features=dense_features, # <--- Pasamos las features densas
        K=10,
        max_candidates=None          # <--- None para evaluar contra TODOS (sin atajos)
    )

# Modelo imagen

In [56]:
def correr_modelo_imagen(img_emb, img_emb_dim):
    fixlen_feature_columns = (
        [
            SparseFeat(
                feat,
                vocabulary_size=sampled_recs_merged[feat].nunique(),
                embedding_dim=8
            )
            for feat in sparse_features
        ]
        +
        [DenseFeat(feat, 1) for feat in dense_features]
        +
        [DenseFeat("img_emb", img_emb_dim)]    
    )

    dnn_feature_columns = fixlen_feature_columns
    linear_feature_columns = fixlen_feature_columns

    feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)

    id2idx = {rid: i for i, rid in enumerate(sampled_recs_merged["review_id"])}

    train_merged[sparse_features] = train_merged[sparse_features].fillna('-1')
    val_merged[sparse_features]   = val_merged[sparse_features].fillna('-1')
    test_merged[sparse_features]  = test_merged[sparse_features].fillna('-1')

    for feat in sparse_features:
        lbe = LabelEncoder()
        all_vals = pd.concat([
            train_merged[feat],
            val_merged[feat],
            test_merged[feat]
        ], axis=0).astype(str)
        lbe.fit(all_vals)

        train_merged[feat] = lbe.transform(train_merged[feat].astype(str))
        val_merged[feat]   = lbe.transform(val_merged[feat].astype(str))
        test_merged[feat]  = lbe.transform(test_merged[feat].astype(str))

    for feat in dense_features:
        train_merged[feat] = pd.to_numeric(train_merged[feat], errors='coerce')
        val_merged[feat]   = pd.to_numeric(val_merged[feat], errors='coerce')
        test_merged[feat]  = pd.to_numeric(test_merged[feat], errors='coerce')

        train_merged[feat] = train_merged[feat].fillna(0)
        val_merged[feat]   = val_merged[feat].fillna(0)
        test_merged[feat]  = test_merged[feat].fillna(0)

    for df in [train_merged, val_merged, test_merged]:
        df["is_recommended"] = df["is_recommended"].astype(int)

    for feat in sparse_features:
        train_merged[feat] = train_merged[feat].astype('int32')
        val_merged[feat]   = val_merged[feat].astype('int32')
        test_merged[feat]  = test_merged[feat].astype('int32')
    for feat in dense_features:
        train_merged[feat] = train_merged[feat].astype('float32')
        val_merged[feat]   = val_merged[feat].astype('float32')
        test_merged[feat]  = test_merged[feat].astype('float32')

    y_train = train_merged[target].values.astype('float32')
    y_val   = val_merged[target].values.astype('float32')
    y_test  = test_merged[target].values.astype('float32')

    train_model_input = {
    name: train_merged[name].values
    for name in feature_names
    if name not in ["img_emb"]
    }
    
    train_model_input["img_emb"]  = train_img_emb.astype("float32")

    val_model_input = {
        name: val_merged[name].values
        for name in feature_names
        if name not in ["img_emb"]
    }
    val_model_input["img_emb"]  = val_img_emb.astype("float32")

    test_model_input = {
        name: test_merged[name].values
        for name in feature_names
        if name not in ["img_emb"]
    }
    test_model_input["img_emb"]  = test_img_emb.astype("float32")

    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = DeepFM(
        linear_feature_columns=linear_feature_columns,
        dnn_feature_columns=dnn_feature_columns,
        task="binary",
        device=device,
    )

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["auc"],
    )

    model.fit(
        train_model_input,
        y_train,
        batch_size=256,
        epochs=10,
        verbose=2,
        validation_data=(val_model_input, y_val),
    )

    pred_test = model.predict(test_model_input, batch_size=256)
    return pred_test, y_test, model, feature_names

In [57]:
def get_split_img_emb(df_split, appid2idx, img_emb):
    """Devuelve matriz [n_rows, img_emb_dim] alineada con df_split.app_id"""
    idxs = []
    for aid in df_split["app_id"]:
        idxs.append(appid2idx.get(aid, None))

    img_dim = img_emb.shape[1]
    arr = np.zeros((len(df_split), img_dim), dtype=img_emb.dtype)

    valid = [i for i, idx in enumerate(idxs) if idx is not None]
    if valid:
        arr[valid] = img_emb[[idxs[i] for i in valid]]

    return arr


def correr_modelo_imagenV2(img_emb, img_emb_dim):
    # 1) Definir columnas (tabulares + imagen)
    fixlen_feature_columns = (
        [
            SparseFeat(
                feat,
                vocabulary_size=sampled_recs_merged[feat].nunique(),
                embedding_dim=8
            )
            for feat in sparse_features
        ]
        +
        [DenseFeat(feat, 1) for feat in dense_features]
        +
        [DenseFeat("img_emb", img_emb_dim)]
    )

    dnn_feature_columns = fixlen_feature_columns
    linear_feature_columns = fixlen_feature_columns

    feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)

    # Mapping app_id -> índice en img_emb
    appid2idx = {aid: i for i, aid in enumerate(df_merged["app_id"].values)}

    # 2) Preprocesar sparse
    train_merged[sparse_features] = train_merged[sparse_features].fillna('-1')
    val_merged[sparse_features]   = val_merged[sparse_features].fillna('-1')
    test_merged[sparse_features]  = test_merged[sparse_features].fillna('-1')

    for feat in sparse_features:
        lbe = LabelEncoder()
        all_vals = pd.concat(
            [train_merged[feat], val_merged[feat], test_merged[feat]],
            axis=0
        ).astype(str)
        lbe.fit(all_vals)

        train_merged[feat] = lbe.transform(train_merged[feat].astype(str))
        val_merged[feat]   = lbe.transform(val_merged[feat].astype(str))
        test_merged[feat]  = lbe.transform(test_merged[feat].astype(str))

    # 3) Preprocesar dense
    for feat in dense_features:
        train_merged[feat] = pd.to_numeric(train_merged[feat], errors='coerce')
        val_merged[feat]   = pd.to_numeric(val_merged[feat], errors='coerce')
        test_merged[feat]  = pd.to_numeric(test_merged[feat], errors='coerce')

        train_merged[feat] = train_merged[feat].fillna(0)
        val_merged[feat]   = val_merged[feat].fillna(0)
        test_merged[feat]  = test_merged[feat].fillna(0)

    for df in [train_merged, val_merged, test_merged]:
        df["is_recommended"] = df["is_recommended"].astype(int)

    for feat in sparse_features:
        train_merged[feat] = train_merged[feat].astype('int32')
        val_merged[feat]   = val_merged[feat].astype('int32')
        test_merged[feat]  = test_merged[feat].astype('int32')

    for feat in dense_features:
        train_merged[feat] = train_merged[feat].astype('float32')
        val_merged[feat]   = val_merged[feat].astype('float32')
        test_merged[feat]  = test_merged[feat].astype('float32')

    # 4) Embeddings de imagen alineados
    train_img_emb = get_split_img_emb(train_merged, appid2idx, img_emb).astype("float32")
    val_img_emb   = get_split_img_emb(val_merged,   appid2idx, img_emb).astype("float32")
    test_img_emb  = get_split_img_emb(test_merged,  appid2idx, img_emb).astype("float32")

    y_train = train_merged[target].values.astype('float32')
    y_val   = val_merged[target].values.astype('float32')
    y_test  = test_merged[target].values.astype('float32')

    # 5) Inputs al modelo
    train_model_input = {
        name: train_merged[name].values
        for name in feature_names
        if name not in ["img_emb"]
    }
    train_model_input["img_emb"] = train_img_emb

    val_model_input = {
        name: val_merged[name].values
        for name in feature_names
        if name not in ["img_emb"]
    }
    val_model_input["img_emb"] = val_img_emb

    test_model_input = {
        name: test_merged[name].values
        for name in feature_names
        if name not in ["img_emb"]
    }
    test_model_input["img_emb"] = test_img_emb

    # 6) DeepFM
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = DeepFM(
        linear_feature_columns=linear_feature_columns,
        dnn_feature_columns=dnn_feature_columns,
        task="binary",
        device=device,
    )

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["auc"],
    )

    model.fit(
        train_model_input,
        y_train,
        batch_size=256,
        epochs=10,
        verbose=2,
        validation_data=(val_model_input, y_val),
    )

    pred_test = model.predict(test_model_input, batch_size=256)
    return pred_test, y_test, model, feature_names


In [58]:
emb_dim = img_emb.shape[1]

y_pred_img, y_test_img, model_img, feature_names_img = correr_modelo_imagenV2(img_emb, emb_dim)
from sklearn.metrics import roc_auc_score, log_loss
print("AUC test:", roc_auc_score(y_test_img, y_pred_img))
print("LogLoss test:", log_loss(y_test_img, y_pred_img))

cpu
Train on 63905 samples, validate on 11649 samples, 250 steps per epoch
Epoch 1/10
3s - loss:  0.4990 - auc:  0.5557 - val_auc:  0.6535
Epoch 2/10
3s - loss:  0.4590 - auc:  0.7110 - val_auc:  0.7037
Epoch 3/10
3s - loss:  0.3747 - auc:  0.7986 - val_auc:  0.7144
Epoch 4/10
4s - loss:  0.2982 - auc:  0.8474 - val_auc:  0.7105
Epoch 5/10
3s - loss:  0.3065 - auc:  0.8536 - val_auc:  0.7082
Epoch 6/10
2s - loss:  0.2940 - auc:  0.8654 - val_auc:  0.6884
Epoch 7/10
3s - loss:  0.2763 - auc:  0.8803 - val_auc:  0.6972
Epoch 8/10
2s - loss:  0.2715 - auc:  0.8838 - val_auc:  0.7081
Epoch 9/10
2s - loss:  0.2548 - auc:  0.8937 - val_auc:  0.7044
Epoch 10/10
2s - loss:  0.2606 - auc:  0.8957 - val_auc:  0.7038
AUC test: 0.6990113133105156
LogLoss test: 0.627580257135064


In [59]:
user_seen = build_user_seen(train_merged, val_merged)
item_cols = ["app_id", "price_final", "average_playtime"]
if "genres" in df_merged.columns:
    item_cols.append("genres")
catalog_df = df_merged[item_cols].drop_duplicates(subset=["app_id"]).copy()

In [60]:
ejecutar_rank = True
id2idx = {rid: i for i, rid in enumerate(sampled_recs_merged["review_id"])}
appid2idx = {app_id: i for i, app_id in enumerate(sampled_recs_merged["app_id"])}
if ejecutar_rank == True:
    all_items = df_merged["app_id"].unique()
    rank_df_only_image = build_candidates_rank_df(
        model=model_img,
        test_df=test_merged,
        user_seen=user_seen,
        all_items=all_items,
        catalog_df=catalog_df,       # <--- Pasamos el catálogo
        id2idx=id2idx,
        text_emb=None,
        feature_names=feature_names_img,
        appid2idx=appid2idx,
        img_emb=img_emb,  # <--- No usamos embeddings de imagen aquí
        dense_features=dense_features, # <--- Pasamos las features densas
        K=10,
        max_candidates=None          # <--- None para evaluar contra TODOS (sin atajos)
    )

# Modelo texto + imagen

In [61]:
def correr_modelo_text_imagen(text_emb, emb_dim, img_emb, img_emb_dim):
    fixlen_feature_columns = (
        [
            SparseFeat(
                feat,
                vocabulary_size=sampled_recs_merged[feat].nunique(),
                embedding_dim=8
            )
            for feat in sparse_features
        ]
        +
        [DenseFeat(feat, 1) for feat in dense_features]
        +
        [DenseFeat("text_emb", emb_dim)]  
        +
        [DenseFeat("img_emb", img_emb_dim)]    
    )

    dnn_feature_columns = fixlen_feature_columns
    linear_feature_columns = fixlen_feature_columns

    feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)

    id2idx = {rid: i for i, rid in enumerate(sampled_recs_merged["review_id"])}

    train_text_emb = get_split_text_emb(train_merged, id2idx, text_emb)
    val_text_emb   = get_split_text_emb(val_merged, id2idx, text_emb)
    test_text_emb  = get_split_text_emb(test_merged, id2idx, text_emb)

    train_merged[sparse_features] = train_merged[sparse_features].fillna('-1')
    val_merged[sparse_features]   = val_merged[sparse_features].fillna('-1')
    test_merged[sparse_features]  = test_merged[sparse_features].fillna('-1')

    for feat in sparse_features:
        lbe = LabelEncoder()
        all_vals = pd.concat([
            train_merged[feat],
            val_merged[feat],
            test_merged[feat]
        ], axis=0).astype(str)
        lbe.fit(all_vals)
        train_merged[feat] = lbe.transform(train_merged[feat].astype(str))
        val_merged[feat]   = lbe.transform(val_merged[feat].astype(str))
        test_merged[feat]  = lbe.transform(test_merged[feat].astype(str))

    for feat in dense_features:
        train_merged[feat] = pd.to_numeric(train_merged[feat], errors='coerce')
        val_merged[feat]   = pd.to_numeric(val_merged[feat], errors='coerce')
        test_merged[feat]  = pd.to_numeric(test_merged[feat], errors='coerce')

        train_merged[feat] = train_merged[feat].fillna(0)
        val_merged[feat]   = val_merged[feat].fillna(0)
        test_merged[feat]  = test_merged[feat].fillna(0)
    for df in [train_merged, val_merged, test_merged]:
        df["is_recommended"] = df["is_recommended"].astype(int)

    for feat in sparse_features:
        train_merged[feat] = train_merged[feat].astype('int32')
        val_merged[feat]   = val_merged[feat].astype('int32')
        test_merged[feat]  = test_merged[feat].astype('int32')
    for feat in dense_features:
        train_merged[feat] = train_merged[feat].astype('float32')
        val_merged[feat]   = val_merged[feat].astype('float32')
        test_merged[feat]  = test_merged[feat].astype('float32')
    train_text_emb = train_text_emb.astype('float32')
    val_text_emb   = val_text_emb.astype('float32')
    test_text_emb  = test_text_emb.astype('float32')
    y_train = train_merged[target].values.astype('float32')
    y_val   = val_merged[target].values.astype('float32')
    y_test  = test_merged[target].values.astype('float32')


    train_model_input = {
        name: train_merged[name].values
        for name in feature_names
        if name not in ["text_emb", "img_emb"]
    }
    train_model_input["text_emb"] = train_text_emb.astype("float32")
    train_model_input["img_emb"]  = train_img_emb.astype("float32")

    val_model_input = {
        name: val_merged[name].values
        for name in feature_names
        if name not in ["text_emb", "img_emb"]
    }
    val_model_input["text_emb"] = val_text_emb.astype("float32")
    val_model_input["img_emb"]  = val_img_emb.astype("float32")

    test_model_input = {
        name: test_merged[name].values
        for name in feature_names
        if name not in ["text_emb", "img_emb"]
    }
    test_model_input["text_emb"] = test_text_emb.astype("float32")
    test_model_input["img_emb"]  = test_img_emb.astype("float32")


    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = DeepFM(
        linear_feature_columns=linear_feature_columns,
        dnn_feature_columns=dnn_feature_columns,
        task="binary",
        device=device,
    )

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["auc"],
    )

    model.fit(
        train_model_input,
        y_train,
        batch_size=256,
        epochs=10,
        verbose=2,
        validation_data=(val_model_input, y_val),
    )

    pred_test = model.predict(test_model_input, batch_size=256)
    return pred_test, y_test, model, feature_names

In [62]:
def get_split_img_emb(df_split, appid2idx, img_emb):
    """Devuelve matriz [n_rows, img_emb_dim] alineada con df_split.app_id"""
    idxs = []
    for aid in df_split["app_id"]:
        idxs.append(appid2idx.get(aid, None))

    img_dim = img_emb.shape[1]
    arr = np.zeros((len(df_split), img_dim), dtype=img_emb.dtype)

    valid = [i for i, idx in enumerate(idxs) if idx is not None]
    if valid:
        arr[valid] = img_emb[[idxs[i] for i in valid]]

    return arr


def correr_modelo_text_imagenV2(text_emb, emb_dim, img_emb, img_emb_dim):
    # 1) Definir columnas de features (tabulares + texto + imagen)
    fixlen_feature_columns = (
        [
            SparseFeat(
                feat,
                vocabulary_size=sampled_recs_merged[feat].nunique(),
                embedding_dim=8
            )
            for feat in sparse_features
        ]
        +
        [DenseFeat(feat, 1) for feat in dense_features]
        +
        [DenseFeat("text_emb", emb_dim)]
        +
        [DenseFeat("img_emb", img_emb_dim)]
    )

    dnn_feature_columns = fixlen_feature_columns
    linear_feature_columns = fixlen_feature_columns

    feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)

    # 2) Mapeos para texto e imagen
    id2idx = {rid: i for i, rid in enumerate(sampled_recs_merged["review_id"])}
    appid2idx = {aid: i for i, aid in enumerate(df_merged["app_id"].values)}

    # 3) Embeddings de TEXTO alineados por split
    train_text_emb = get_split_text_emb(train_merged, id2idx, text_emb)
    val_text_emb   = get_split_text_emb(val_merged, id2idx, text_emb)
    test_text_emb  = get_split_text_emb(test_merged, id2idx, text_emb)

    # 4) Embeddings de IMAGEN alineados por split
    train_img_emb = get_split_img_emb(train_merged, appid2idx, img_emb)
    val_img_emb   = get_split_img_emb(val_merged, appid2idx, img_emb)
    test_img_emb  = get_split_img_emb(test_merged, appid2idx, img_emb)

    # 5) Preprocesado de sparse y dense (igual que antes)
    train_merged[sparse_features] = train_merged[sparse_features].fillna('-1')
    val_merged[sparse_features]   = val_merged[sparse_features].fillna('-1')
    test_merged[sparse_features]  = test_merged[sparse_features].fillna('-1')

    for feat in sparse_features:
        lbe = LabelEncoder()
        all_vals = pd.concat([
            train_merged[feat],
            val_merged[feat],
            test_merged[feat]
        ], axis=0).astype(str)
        lbe.fit(all_vals)
        train_merged[feat] = lbe.transform(train_merged[feat].astype(str))
        val_merged[feat]   = lbe.transform(val_merged[feat].astype(str))
        test_merged[feat]  = lbe.transform(test_merged[feat].astype(str))

    for feat in dense_features:
        train_merged[feat] = pd.to_numeric(train_merged[feat], errors='coerce')
        val_merged[feat]   = pd.to_numeric(val_merged[feat], errors='coerce')
        test_merged[feat]  = pd.to_numeric(test_merged[feat], errors='coerce')

        train_merged[feat] = train_merged[feat].fillna(0)
        val_merged[feat]   = val_merged[feat].fillna(0)
        test_merged[feat]  = test_merged[feat].fillna(0)

    for df in [train_merged, val_merged, test_merged]:
        df["is_recommended"] = df["is_recommended"].astype(int)

    for feat in sparse_features:
        train_merged[feat] = train_merged[feat].astype('int32')
        val_merged[feat]   = val_merged[feat].astype('int32')
        test_merged[feat]  = test_merged[feat].astype('int32')

    for feat in dense_features:
        train_merged[feat] = train_merged[feat].astype('float32')
        val_merged[feat]   = val_merged[feat].astype('float32')
        test_merged[feat]  = test_merged[feat].astype('float32')

    train_text_emb = train_text_emb.astype('float32')
    val_text_emb   = val_text_emb.astype('float32')
    test_text_emb  = test_text_emb.astype('float32')

    train_img_emb = train_img_emb.astype('float32')
    val_img_emb   = val_img_emb.astype('float32')
    test_img_emb  = test_img_emb.astype('float32')

    y_train = train_merged[target].values.astype('float32')
    y_val   = val_merged[target].values.astype('float32')
    y_test  = test_merged[target].values.astype('float32')

    # 6) Inputs al modelo
    train_model_input = {
        name: train_merged[name].values
        for name in feature_names
        if name not in ["text_emb", "img_emb"]
    }
    train_model_input["text_emb"] = train_text_emb
    train_model_input["img_emb"]  = train_img_emb

    val_model_input = {
        name: val_merged[name].values
        for name in feature_names
        if name not in ["text_emb", "img_emb"]
    }
    val_model_input["text_emb"] = val_text_emb
    val_model_input["img_emb"]  = val_img_emb

    test_model_input = {
        name: test_merged[name].values
        for name in feature_names
        if name not in ["text_emb", "img_emb"]
    }
    test_model_input["text_emb"] = test_text_emb
    test_model_input["img_emb"]  = test_img_emb

    # 7) DeepFM
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = DeepFM(
        linear_feature_columns=linear_feature_columns,
        dnn_feature_columns=dnn_feature_columns,
        task="binary",
        device=device,
    )

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["auc"],
    )

    model.fit(
        train_model_input,
        y_train,
        batch_size=256,
        epochs=10,
        verbose=2,
        validation_data=(val_model_input, y_val),
    )

    pred_test = model.predict(test_model_input, batch_size=256)
    return pred_test, y_test, model, feature_names


In [63]:
text_emb = text_emb_all_mini
emb_dim = text_emb.shape[1]
img_emb = img_emb
img_emb_dim = img_emb.shape[1]

y_pred_full, y_test_full, model_full, feature_names_full = correr_modelo_text_imagenV2(text_emb, emb_dim,img_emb, img_emb_dim)
from sklearn.metrics import roc_auc_score, log_loss
print("AUC test:", roc_auc_score(y_test_full, y_pred_full))
print("LogLoss test:", log_loss(y_test_full, y_pred_full))

cpu
Train on 63905 samples, validate on 11649 samples, 250 steps per epoch
Epoch 1/10
4s - loss:  0.4405 - auc:  0.5436 - val_auc:  0.6369
Epoch 2/10
3s - loss:  0.3600 - auc:  0.7193 - val_auc:  0.7064
Epoch 3/10
3s - loss:  0.3138 - auc:  0.8174 - val_auc:  0.7054
Epoch 4/10
3s - loss:  0.3125 - auc:  0.8429 - val_auc:  0.6986
Epoch 5/10
3s - loss:  0.2831 - auc:  0.8661 - val_auc:  0.7073
Epoch 6/10
3s - loss:  0.2750 - auc:  0.8739 - val_auc:  0.7077
Epoch 7/10
4s - loss:  0.2695 - auc:  0.8820 - val_auc:  0.7110
Epoch 8/10
4s - loss:  0.2612 - auc:  0.8898 - val_auc:  0.6972
Epoch 9/10
3s - loss:  0.2757 - auc:  0.8820 - val_auc:  0.6925
Epoch 10/10
3s - loss:  0.2800 - auc:  0.8861 - val_auc:  0.6981
AUC test: 0.6917307687090963
LogLoss test: 0.7070565186161339


In [64]:
user_seen = build_user_seen(train_merged, val_merged)
item_cols = ["app_id", "price_final", "average_playtime"]
if "genres" in df_merged.columns:
    item_cols.append("genres")
catalog_df = df_merged[item_cols].drop_duplicates(subset=["app_id"]).copy()

In [65]:
all_items = df_merged["app_id"].unique()
id2idx = {rid: i for i, rid in enumerate(sampled_recs_merged["review_id"])}
appid2idx = {app_id: i for i, app_id in enumerate(sampled_recs_merged["app_id"])}
ejecutar_rank = True
if ejecutar_rank == True:
    rank_df_image_texto = build_candidates_rank_df(
        model=model_full,
        test_df=test_merged,
        user_seen=user_seen,
        all_items=all_items,
        catalog_df=catalog_df,       # <--- Pasamos el catálogo
        id2idx=id2idx,
        text_emb=text_emb,
        feature_names=feature_names_full,
        appid2idx=appid2idx,
        img_emb=img_emb,  # <--- No usamos embeddings de imagen aquí
        dense_features=dense_features, # <--- Pasamos las features densas
        K=10,
        max_candidates=None          # <--- None para evaluar contra TODOS (sin atajos)
    )

# Modelo Solo

In [66]:
def correr_modelo_solo():
    fixlen_feature_columns = (
        [
            SparseFeat(
                feat,
                vocabulary_size=sampled_recs_merged[feat].nunique(),
                embedding_dim=8
            )
            for feat in sparse_features
        ]
        +
        [DenseFeat(feat, 1) for feat in dense_features]
      
    )

    dnn_feature_columns = fixlen_feature_columns
    linear_feature_columns = fixlen_feature_columns

    feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)

    id2idx = {rid: i for i, rid in enumerate(sampled_recs_merged["review_id"])}

    train_merged[sparse_features] = train_merged[sparse_features].fillna('-1')
    val_merged[sparse_features]   = val_merged[sparse_features].fillna('-1')
    test_merged[sparse_features]  = test_merged[sparse_features].fillna('-1')

    for feat in sparse_features:
        lbe = LabelEncoder()
        all_vals = pd.concat([
            train_merged[feat],
            val_merged[feat],
            test_merged[feat]
        ], axis=0).astype(str)
        lbe.fit(all_vals)
        train_merged[feat] = lbe.transform(train_merged[feat].astype(str))
        val_merged[feat]   = lbe.transform(val_merged[feat].astype(str))
        test_merged[feat]  = lbe.transform(test_merged[feat].astype(str))

    for feat in dense_features:
        train_merged[feat] = pd.to_numeric(train_merged[feat], errors='coerce')
        val_merged[feat]   = pd.to_numeric(val_merged[feat], errors='coerce')
        test_merged[feat]  = pd.to_numeric(test_merged[feat], errors='coerce')

        train_merged[feat] = train_merged[feat].fillna(0)
        val_merged[feat]   = val_merged[feat].fillna(0)
        test_merged[feat]  = test_merged[feat].fillna(0)
    for df in [train_merged, val_merged, test_merged]:
        df["is_recommended"] = df["is_recommended"].astype(int)

    for feat in sparse_features:
        train_merged[feat] = train_merged[feat].astype('int32')
        val_merged[feat]   = val_merged[feat].astype('int32')
        test_merged[feat]  = test_merged[feat].astype('int32')
    for feat in dense_features:
        train_merged[feat] = train_merged[feat].astype('float32')
        val_merged[feat]   = val_merged[feat].astype('float32')
        test_merged[feat]  = test_merged[feat].astype('float32')

    y_train = train_merged[target].values.astype('float32')
    y_val   = val_merged[target].values.astype('float32')
    y_test  = test_merged[target].values.astype('float32')

    train_model_input = {
        name: train_merged[name].values
        for name in feature_names
    }

    val_model_input = {
        name: val_merged[name].values
        for name in feature_names
    }

    test_model_input = {
        name: test_merged[name].values
        for name in feature_names
    }

    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = DeepFM(
        linear_feature_columns=linear_feature_columns,
        dnn_feature_columns=dnn_feature_columns,
        task="binary",
        device=device,
    )

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["auc"],
    )

    model.fit(
        train_model_input,
        y_train,
        batch_size=256,
        epochs=10,
        verbose=2,
        validation_data=(val_model_input, y_val),
    )

    pred_test = model.predict(test_model_input, batch_size=256)
    return pred_test, y_test, model, feature_names

In [67]:
y_pred_solo, y_test_solo, model_solo, feature_names_solo = correr_modelo_solo()
from sklearn.metrics import roc_auc_score, log_loss
print("AUC test:", roc_auc_score(y_test_solo, y_pred_solo))
print("LogLoss test:", log_loss(y_test_solo, y_pred_solo))

cpu
Train on 63905 samples, validate on 11649 samples, 250 steps per epoch
Epoch 1/10
2s - loss:  0.5505 - auc:  0.5420 - val_auc:  0.6521
Epoch 2/10
1s - loss:  0.4666 - auc:  0.7270 - val_auc:  0.7112
Epoch 3/10
1s - loss:  0.3702 - auc:  0.8045 - val_auc:  0.6954
Epoch 4/10
2s - loss:  0.3222 - auc:  0.8450 - val_auc:  0.7092
Epoch 5/10
3s - loss:  0.3590 - auc:  0.8518 - val_auc:  0.6975
Epoch 6/10
2s - loss:  0.2761 - auc:  0.8821 - val_auc:  0.7149
Epoch 7/10
2s - loss:  0.2668 - auc:  0.8916 - val_auc:  0.7057
Epoch 8/10
2s - loss:  0.2416 - auc:  0.9055 - val_auc:  0.7097
Epoch 9/10
2s - loss:  0.2496 - auc:  0.9047 - val_auc:  0.7055
Epoch 10/10
2s - loss:  0.2553 - auc:  0.9060 - val_auc:  0.7026
AUC test: 0.6954398030813159
LogLoss test: 0.5433203057884005


In [68]:
user_seen = build_user_seen(train_merged, val_merged)
item_cols = ["app_id", "price_final", "average_playtime"]
if "genres" in df_merged.columns:
    item_cols.append("genres")
catalog_df = df_merged[item_cols].drop_duplicates(subset=["app_id"]).copy()

In [69]:
all_items = df_merged["app_id"].unique()
id2idx = {rid: i for i, rid in enumerate(sampled_recs_merged["review_id"])}
appid2idx = {app_id: i for i, app_id in enumerate(sampled_recs_merged["app_id"])}
ejecutar_rank = True
if ejecutar_rank == True:
    rank_df_solo = build_candidates_rank_df(
        model=model_solo,
        test_df=test_merged,
        user_seen=user_seen,
        all_items=all_items,
        catalog_df=catalog_df,       # <--- Pasamos el catálogo
        id2idx=id2idx,
        text_emb=None,
        feature_names=feature_names_solo,
        appid2idx=appid2idx,
        img_emb=None,  # <--- No usamos embeddings de imagen aquí
        dense_features=dense_features, # <--- Pasamos las features densas
        K=10,
        max_candidates=None          # <--- None para evaluar contra TODOS (sin atajos)
    )

# Funciones de Metricas

In [70]:
import numpy as np

def dcg_at_k(labels, k):
    labels = np.asarray(labels)[:k]
    if labels.size == 0:
        return 0.0
    gains = (2 ** labels - 1).astype(float)
    discounts = np.log2(np.arange(2, gains.size + 2))
    return float(np.sum(gains / discounts))

def ndcg_at_k(labels, k):
    labels = np.asarray(labels)
    ideal = np.sort(labels)[::-1]
    ideal_dcg = dcg_at_k(ideal, k)
    if ideal_dcg == 0.0:
        return 0.0
    return dcg_at_k(labels, k) / ideal_dcg

def average_precision_at_k(labels, k):
    labels = np.asarray(labels)[:k]
    if labels.size == 0:
        return 0.0
    hits = 0
    sum_prec = 0.0
    for i, y in enumerate(labels, start=1):
        if y == 1:
            hits += 1
            sum_prec += hits / i
    if hits == 0:
        return 0.0
    return sum_prec / min(hits, k)

def diversity_at_k(rank_df, k=10):
    divs = []

    for user_id, group in rank_df.groupby("user_id"):
        # Top-K por score
        topk = group.sort_values("score", ascending=False).head(k)

        if "genres" not in topk.columns:
            continue

        genres_list = topk["genres"].fillna("").astype(str).tolist()

        genres_split = set()
        for g in genres_list:
            for token in str(g).split(";"):
                token = token.strip()
                if token:
                    genres_split.add(token)

        divs.append(len(genres_split))

    return float(np.mean(divs)) if divs else 0.0

In [71]:
def macro_metrics_at_k(rank_df, k=10):
    users = rank_df["user_id"].unique()

    precs, recs, f1s, ndcgs = [], [], [], []
    hit_list, ap_list = [], []
    divs = []

    for u in users:
        df_u = rank_df[rank_df["user_id"] == u]
        if df_u.empty:
            continue

        topk = df_u.head(k)
        labels = topk["label"].astype(int).to_numpy()
        total_rel = int(df_u["label"].sum())

        hits_k = int(labels.sum())
        prec = hits_k / k if k > 0 else 0.0
        rec = hits_k / total_rel if total_rel > 0 else 0.0
        if prec + rec > 0:
            f1 = 2 * prec * rec / (prec + rec)
        else:
            f1 = 0.0

        ndcg = ndcg_at_k(labels, k)

        hit = 1.0 if hits_k > 0 else 0.0

        ap = average_precision_at_k(labels, k)
        
        if "genres" in topk.columns:
            genres_list = topk["genres"].fillna("").astype(str).tolist()
            genres_split = set()
            for g in genres_list:
                for token in str(g).split(";"):
                    token = token.strip()
                    if token:
                        genres_split.add(token)
            div = len(genres_split)  
        else:
            div = np.nan

        precs.append(prec)
        recs.append(rec)
        f1s.append(f1)
        ndcgs.append(ndcg)
        hit_list.append(hit)
        ap_list.append(ap)
        divs.append(div)

    metrics = {
        "Precision@K": float(np.nanmean(precs)) if precs else 0.0,
        "Recall@K": float(np.nanmean(recs)) if recs else 0.0,
        "F1@K": float(np.nanmean(f1s)) if f1s else 0.0,
        "NDCG@K": float(np.nanmean(ndcgs)) if ndcgs else 0.0,
        "HitRate@K": float(np.nanmean(hit_list)) if hit_list else 0.0,
        "MAP@K": float(np.nanmean(ap_list)) if ap_list else 0.0,
        "Diversity@K": float(np.nanmean(divs)) if divs else 0.0,
        
    }
    return metrics

# Métricas Resultados

In [72]:
metrics_k_only_allmini = macro_metrics_at_k(rank_df_only_allmini, k=10)
print(f"Resultados Top-10 (macro):")
for name, value in metrics_k_only_allmini.items():
    print(f"{name}: {value:.4f}")

Resultados Top-10 (macro):
Precision@K: 0.0165
Recall@K: 0.1410
F1@K: 0.0289
NDCG@K: 0.0524
HitRate@K: 0.1568
MAP@K: 0.0228
Diversity@K: 12.0255


In [73]:
metrics_k_only_mpnet = macro_metrics_at_k(rank_df_only_mpnet, k=10)
print(f"Resultados Top-10 (macro):")
for name, value in metrics_k_only_mpnet.items():
    print(f"{name}: {value:.4f}")

Resultados Top-10 (macro):
Precision@K: 0.0319
Recall@K: 0.2636
F1@K: 0.0557
NDCG@K: 0.2440
HitRate@K: 0.2902
MAP@K: 0.2296
Diversity@K: 9.3640


In [74]:
metrics_k_only_image = macro_metrics_at_k(rank_df_only_image, k=10)
print(f"Resultados Top-10 (macro):")
for name, value in metrics_k_only_image.items():
    print(f"{name}: {value:.4f}")

Resultados Top-10 (macro):
Precision@K: 0.0060
Recall@K: 0.0442
F1@K: 0.0101
NDCG@K: 0.0240
HitRate@K: 0.0553
MAP@K: 0.0141
Diversity@K: 11.2830


In [75]:
metrics_k_full = macro_metrics_at_k(rank_df_image_texto, k=10)
print(f"Resultados Top-10 (macro):")
for name, value in metrics_k_full.items():
    print(f"{name}: {value:.4f}")

Resultados Top-10 (macro):
Precision@K: 0.0005
Recall@K: 0.0049
F1@K: 0.0010
NDCG@K: 0.0030
HitRate@K: 0.0054
MAP@K: 0.0023
Diversity@K: 9.0167


In [76]:
metrics_k_solo = macro_metrics_at_k(rank_df_solo, k=10)
print(f"Resultados Top-10 (macro):")
for name, value in metrics_k_solo.items():
    print(f"{name}: {value:.4f}")

Resultados Top-10 (macro):
Precision@K: 0.0214
Recall@K: 0.1702
F1@K: 0.0367
NDCG@K: 0.0905
HitRate@K: 0.1944
MAP@K: 0.0567
Diversity@K: 13.9162


In [78]:
rows = []

metrics_k_solo["modelo"] = "solo_tabular"
rows.append(metrics_k_solo)

metrics_k_only_allmini["modelo"] = "allmini_texto"
rows.append(metrics_k_only_allmini)

metrics_k_only_mpnet["modelo"] = "mpnet_texto"
rows.append(metrics_k_only_mpnet)

metrics_k_only_image["modelo"] = "imagen"
rows.append(metrics_k_only_image)

metrics_k_full["modelo"] = "texto_imagen"
rows.append(metrics_k_full)

df_all = pd.DataFrame(rows)
df_all.to_csv("metrics_todos_modelos_k10.csv", index=False)

# Resumen Metricas

In [77]:
auc_allmini = roc_auc_score(y_test_allmini, y_pred_allmini)
auc_mpnet = roc_auc_score(y_test_mpnet, y_pred_mpnet)
auc_img = roc_auc_score(y_test_img, y_pred_img)
auc_full = roc_auc_score(y_test_full, y_pred_full)
auc_solo = roc_auc_score(y_test_solo, y_pred_solo)

logloss_allmini = log_loss(y_test_allmini, y_pred_allmini)
logloss_mpnet = log_loss(y_test_mpnet, y_pred_mpnet)
logloss_img = log_loss(y_test_img, y_pred_img)
logloss_full = log_loss(y_test_full, y_pred_full)
logloss_solo = log_loss(y_test_solo, y_pred_solo)

# Tabla resumen
import pandas as pd
results_summary = pd.DataFrame({
    "Model": [
        "Only All-MiniLM",
        "Only MPNet",
        "Only Image Embeddings",
        "Text + Image Embeddings",
        "No Embeddings"
    ],
    "AUC": [
        auc_allmini,
        auc_mpnet,
        auc_img,
        auc_full,
        auc_solo
    ],
    "LogLoss": [
        logloss_allmini,
        logloss_mpnet,
        logloss_img,
        logloss_full,
        logloss_solo
    ]
})

results_summary

,Model,AUC,LogLoss
0,Only All-MiniLM,0.700546,0.565809
1,Only MPNet,0.708063,0.610823
2,Only Image Embeddings,0.699011,0.627580
3,Text + Image Embeddings,0.691731,0.707057
4,No Embeddings,0.695440,0.543320
